# TriDep — 03 · Demo
Run cells **A → E** in order for a live demo.

**Prerequisite:** `02_train_evaluate.ipynb` must have been run at least once so the demo bundle exists on Drive.

In [ ]:
from pathlib import Path
import os, shutil

try:
    from google.colab import drive
    drive.mount('/content/drive')
    shutil.unpack_archive('/content/drive/MyDrive/DAIC/demo_bundle.zip',
                          '/content/demo', 'zip')
    print('Demo bundle ready (Google Drive).')
except ImportError:
    base_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    saved_bundle = base_dir / 'saved_models' / 'demo_bundle.zip'
    local_demo = base_dir / 'saved_models' / 'demo_bundle'
    local_demo.mkdir(parents=True, exist_ok=True)
    if saved_bundle.exists() and not (local_demo / 'fusion_model.keras').exists():
        shutil.unpack_archive(str(saved_bundle), str(local_demo), 'zip')
    print('Demo bundle ready (Local):', local_demo)


Mounted at /content/drive
Demo bundle ready.


## Run cells A → E in order
Each cell does one job and saves its output to `/tmp/`. If a cell crashes, re-run only that cell.

In [ ]:
# ── STEP A: Upload your files ──────────────────────────────
# Set file paths below OR leave empty to get a file dialog.
AUDIO_PATH = ""   # .wav / .mp3
VIDEO_PATH = ""   # .mp4 / .avi  (optional)
IMAGE_PATH = ""   # .jpg / .png  (optional fallback)
# ──────────────────────────────────────────────────────────

import os, tempfile
from pathlib import Path

_demo_audio = AUDIO_PATH or None
_demo_video = VIDEO_PATH or None
_demo_image = IMAGE_PATH or None

TMP_DIR = Path(tempfile.gettempdir()) / "tridep_demo"
TMP_DIR.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import files as _cf
    IS_COLAB = True
except ImportError:
    _cf = None
    IS_COLAB = False

if not _demo_audio and not _demo_video and not _demo_image:
    if IS_COLAB and _cf is not None:
        print("Select files in the Colab upload dialog:")
        _up = _cf.upload()
        for _fname, _data in _up.items():
            _ext  = Path(_fname).suffix.lower()
            _dest = str(TMP_DIR / _fname)
            Path(_dest).write_bytes(_data)
            if _ext in ('.wav','.mp3','.ogg','.flac','.m4a'):
                _demo_audio = _dest; print(f"  audio  -> {_dest}")
            elif _ext in ('.mp4','.avi','.mov','.mkv','.webm'):
                _demo_video = _dest; print(f"  video  -> {_dest}")
            elif _ext in ('.jpg','.jpeg','.png','.bmp','.webp'):
                _demo_image = _dest; print(f"  image  -> {_dest}")
    else:
        # Local Windows file picker dialog
        try:
            import tkinter as tk
            from tkinter import filedialog
            root = tk.Tk()
            root.withdraw()
            root.attributes('-topmost', True)
            print("Opening file selection dialog for audio...")
            selected = filedialog.askopenfilename(
                title="Select Audio File for TriDep",
                filetypes=[("Audio files", "*.wav *.mp3 *.ogg *.m4a"), ("All files", "*.*")]
            )
            root.destroy()
            if selected:
                _demo_audio = selected
        except Exception as e:
            print("File picker dialog skipped:", e)

if not _demo_audio:
    print("\nℹ️ Note: Set AUDIO_PATH at the top of this cell to test a specific audio file.")
    print("   👉 OR scroll directly to the Gradio Web App cell below to test files via drag-and-drop!")
else:
    print("\nReady:")
    print(f"  Audio : {_demo_audio}")
    print(f"  Video : {_demo_video or '(none)'}")
    print(f"  Image : {_demo_image or '(none)'}")
    print("\nRun STEP B next.")


Select ALL files at once in the dialog (audio + video/image):


Saving WhatsApp Video 2026-06-15 at 12.01.34 PM.mp4 to WhatsApp Video 2026-06-15 at 12.01.34 PM.mp4
Saving 300_AUDIO.wav to 300_AUDIO.wav
  video  -> /tmp/WhatsApp Video 2026-06-15 at 12.01.34 PM.mp4
  audio  -> /tmp/300_AUDIO.wav

Ready:
  Audio : /tmp/300_AUDIO.wav
  Video : /tmp/WhatsApp Video 2026-06-15 at 12.01.34 PM.mp4
  Image : (none)

Run STEP B next.


In [ ]:
# ── STEP B: Text features — Whisper -> SBERT -> /tmp/t_vec.npy ──
# Only the first 3 minutes of audio are transcribed to avoid OOM.
# Whisper is deleted before SBERT loads.
import re, gc, sys, subprocess, numpy as np, librosa, soundfile as sf

def _pip(p): subprocess.check_call([sys.executable,"-m","pip","install","-q",p])
_pip("openai-whisper")
_pip("soundfile")

SAMPLE_RATE        = 16_000
TRANSCRIPT_MAX_SEC = 180    # first 3 min is plenty for SBERT context

print(f"Clipping audio to first {TRANSCRIPT_MAX_SEC}s ...")
_clip, _sr = librosa.load(_demo_audio, sr=None, mono=True, duration=TRANSCRIPT_MAX_SEC)
if _sr != SAMPLE_RATE:
    _clip = librosa.resample(_clip.astype("float32"), orig_sr=_sr, target_sr=SAMPLE_RATE)
_clip_path = "/tmp/_whisper_clip.wav"
sf.write(_clip_path, _clip.astype("float32"), SAMPLE_RATE)
del _clip; gc.collect()
print(f"  Clip ready: {TRANSCRIPT_MAX_SEC}s @ 16kHz")

import whisper as _wh
print("Loading Whisper 'tiny' ...")
_wm = _wh.load_model("tiny")
print("Transcribing ...")
_res = _wm.transcribe(_clip_path, fp16=False)
raw_text = _res["text"].strip()
print(f'Transcript: "{raw_text[:160]}{"..." if len(raw_text)>160 else ""}"')

# Delete Whisper before loading SBERT
del _wm, _res; gc.collect()
try:
    import torch; torch.cuda.empty_cache()
except: pass

# Same text cleaning as Stage 2 training pipeline
_c = re.sub(r"<[^>]+>",     " ", raw_text)
_c = re.sub(r"\b(\w+)V\b",  r"\1", _c)
_c = re.sub(r"_",            " ", _c)
_c = re.sub(r"\s+",         " ", _c).strip()
_sents = [s.strip() for s in _c.replace("?",".").split(".") if s.strip()]
if not _sents: _sents = [_c or "no speech"]
print(f"Sentences: {len(_sents)}")

from sentence_transformers import SentenceTransformer
print("Loading Sentence-BERT ...")
_sb   = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
t_vec = _sb.encode(_sents, show_progress_bar=False,
                   convert_to_numpy=True).mean(axis=0).astype("float32")
del _sb; gc.collect()

np.save("/tmp/t_vec.npy", t_vec)
print(f"\nText vector -> /tmp/t_vec.npy  {t_vec.shape}  OK")
print("Run STEP C next.")

Clipping audio to first 180s ...
  Clip ready: 180s @ 16kHz
Loading Whisper 'tiny' ...


100%|██████████████████████████████████████| 72.1M/72.1M [00:00<00:00, 151MiB/s]


Transcribing ...
Transcript: "that's what you call your body. So if you can move around a little bit, we'll make sure that the connect is recognizing you. So you just move your hand, go it a..."
Sentences: 56
Loading Sentence-BERT ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Text vector -> /tmp/t_vec.npy  (768,)  OK
Run STEP C next.


In [ ]:
# ── STEP C: Audio features — Wav2Vec2 -> /tmp/a_vec.npy ────────
import gc, numpy as np, librosa, torch

SAMPLE_RATE  = 16_000
WINDOW_LEN   = SAMPLE_RATE * 8   # 8-second windows (same as training)

# Use fewer windows on CPU to avoid 10-min hang; GPU can handle all 20
_dev = "cuda" if torch.cuda.is_available() else "cpu"
MAX_WINDOWS  = 20 if _dev == "cuda" else 5
MAX_LOAD_SEC = MAX_WINDOWS * 8 + 30

print(f"Device: {_dev}  |  Max windows: {MAX_WINDOWS}")
print(f"Loading first {MAX_LOAD_SEC}s of audio ...")
_sp, _sr = librosa.load(_demo_audio, sr=None, mono=True, duration=MAX_LOAD_SEC)
if _sr != SAMPLE_RATE:
    _sp = librosa.resample(_sp.astype("float32"), orig_sr=_sr, target_sr=SAMPLE_RATE)
_sp = _sp.astype("float32")
print(f"  Loaded: {len(_sp)/SAMPLE_RATE:.1f}s @ 16kHz")

_wins = [_sp[i:i+WINDOW_LEN]
         for i in range(0, len(_sp)-WINDOW_LEN+1, WINDOW_LEN)][:MAX_WINDOWS]
print(f"  Windows: {len(_wins)} x 8s")

if _wins:
    from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor
    print(f"  Loading Wav2Vec2 ...")
    _fe  = Wav2Vec2FeatureExtractor.from_pretrained("facebook/wav2vec2-base-960h")
    _w2v = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h").to(_dev).eval()

    # process one window at a time — avoids padding overhead and OOM
    _embs = []
    for _i, _win in enumerate(_wins):
        print(f"  Window {_i+1}/{len(_wins)} ...", end="\r")
        with torch.no_grad():
            _inp = _fe([_win], sampling_rate=SAMPLE_RATE, return_tensors="pt", padding=True)
            _hid = _w2v(_inp.input_values.to(_dev)).last_hidden_state
            _embs.append(_hid.mean(dim=1).cpu().numpy())
    print()

    _emb  = np.concatenate(_embs, axis=0)   # (N, 768)
    a_vec = np.concatenate([_emb.mean(0), _emb.std(0)]).astype("float32")
    del _w2v, _fe, _hid, _embs, _emb; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
else:
    print("  Audio shorter than 8s — zero vector used.")
    a_vec = np.zeros(1536, dtype="float32")

np.save("/tmp/a_vec.npy", a_vec)
print(f"\nAudio vector -> /tmp/a_vec.npy  {a_vec.shape}  OK")
print("Run STEP D next.")


Device: cpu  |  Max windows: 5
Loading first 70s of audio ...
  Loaded: 70.0s @ 16kHz
  Windows: 5 x 8s
  Loading Wav2Vec2 ...


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.60k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  378MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.




Audio vector -> /tmp/a_vec.npy  (1536,)  OK
Run STEP D next.


In [ ]:
# ── STEP D: Video/Image features — py-feat AUs -> /tmp/f_vec.npy ──
import gc, sys, subprocess, importlib, numpy as np, builtins, os

TRAIN_AU_COLS = [
    'AU01_r','AU02_r','AU04_r','AU05_r','AU06_r','AU09_r','AU10_r',
    'AU12_r','AU14_r','AU15_r','AU17_r','AU20_r','AU25_r','AU26_r',
    'AU04_c','AU12_c','AU15_c','AU23_c','AU28_c','AU45_c'
]
_MAP = {
    'AU01_r':'AU01','AU02_r':'AU02','AU04_r':'AU04','AU05_r':'AU05',
    'AU06_r':'AU06','AU09_r':'AU09','AU10_r':'AU10','AU12_r':'AU12',
    'AU14_r':'AU14','AU15_r':'AU15','AU17_r':'AU17','AU20_r':'AU20',
    'AU25_r':'AU25','AU26_r':'AU26',
    'AU04_c':'AU04','AU12_c':'AU12','AU15_c':'AU15',
    'AU23_c':'AU23','AU28_c':'AU28','AU45_c':'AU43',
}

# ── NumPy 2.0 compat shims ────────────────────────────────────────────────────
for _attr, _val in {
    'bool': builtins.bool, 'int': builtins.int, 'float': builtins.float,
    'complex': builtins.complex, 'object': builtins.object, 'str': builtins.str,
    'long': builtins.int, 'unicode': builtins.str,
}.items():
    if not hasattr(np, _attr):
        setattr(np, _attr, _val)
if not hasattr(np, 'ComplexWarning'):
    np.ComplexWarning = getattr(getattr(np, 'exceptions', None), 'ComplexWarning', builtins.Warning)
if not hasattr(np, 'VisibleDeprecationWarning'):
    np.VisibleDeprecationWarning = getattr(getattr(np, 'exceptions', None), 'VisibleDeprecationWarning', builtins.Warning)

# ── scipy compat shims ────────────────────────────────────────────────────────
import scipy.stats as _ss, scipy.integrate as _si
if not hasattr(_ss, 'binom_test'):
    _ss.binom_test = lambda k, n=None, p=0.5, alternative='two-sided': \
        _ss.binomtest(k, n, p, alternative=alternative).pvalue
if not hasattr(_si, 'simps'):
    _si.simps = _si.simpson

# ── ensure av (PyAV) is installed — needed for torchvision.io.read_video shim ─
try:
    import av as _av_pkg
except ImportError:
    print('Installing av ...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'av', '-q'],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    import av as _av_pkg

# ── torchvision.io.read_video removed in 0.19+ — shim using av ───────────────
import torch, torchvision.io as _tvio
if not hasattr(_tvio, 'read_video'):
    def _read_video(filename, start_pts=0, end_pts=None, pts_unit='pts', output_format='THWC'):
        frames = []
        with _av_pkg.open(filename) as _c:
            _vs  = _c.streams.video[0]
            _fps = float(_vs.average_rate) if _vs.average_rate else 25.0
            for _f in _c.decode(video=0):
                frames.append(torch.from_numpy(_f.to_ndarray(format='rgb24')))
        vframes = torch.stack(frames) if frames else torch.zeros(0,1,1,3,dtype=torch.uint8)
        return vframes, torch.zeros(0), {'video_fps': _fps}
    _tvio.read_video = _read_video

# ── Install py-feat 0.5.1 only if not already present ────────────────────────
_PYFEAT_DIR = '/tmp/pyfeat_install'
if os.path.exists(f'{_PYFEAT_DIR}/feat'):
    print('py-feat already installed — skipping ✅')
else:
    print('Installing py-feat==0.5.1 (~30s) ...')
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', 'py-feat==0.5.1',
         f'--target={_PYFEAT_DIR}', '-q'],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    print('  done ✅')

if _PYFEAT_DIR not in sys.path:
    sys.path.insert(0, _PYFEAT_DIR)
for _m in list(sys.modules.keys()):
    if _m == 'feat' or _m.startswith('feat.'):
        del sys.modules[_m]
importlib.invalidate_caches()

def _get_detector():
    try:
        from feat import Detector
        return Detector
    except Exception:
        pass
    try:
        return importlib.import_module('feat.detector').Detector
    except Exception as _e:
        raise ImportError(f"Detector unavailable: {_e}")

_vis  = _demo_video or _demo_image
f_vec = None

if _vis:
    try:
        Detector = _get_detector()
        print('Loading Detector (downloads AU models on first run) ...')
        _det = Detector(device="cuda" if torch.cuda.is_available() else "cpu")

        if _demo_video:
            print(f"Detecting AUs in video: {_demo_video} ...")
            _fea = _det.detect_video(_demo_video)
        else:
            print(f"Detecting AUs in image: {_demo_image} ...")
            _fea = _det.detect_image(_demo_image)

        _au = _fea.aus.dropna(how="all").reset_index(drop=True)
        if len(_au) == 0:
            raise ValueError("No face detected.")
        print(f"  Frames with face: {len(_au)}")

        _arr = np.zeros((len(_au), 20), dtype="float32")
        for _i, _tc in enumerate(TRAIN_AU_COLS):
            _pc = _MAP.get(_tc)
            if _pc and _pc in _au.columns:
                _arr[:, _i] = _au[_pc].fillna(0).values.astype("float32")

        if _arr.shape[0] > 1:
            _arr = (_arr - _arr.mean(0,keepdims=True)) / (_arr.std(0,keepdims=True)+1e-8)
        f_vec = _arr.mean(axis=0).astype("float32")
        del _det, _fea, _au, _arr; gc.collect()
        print("  py-feat ✅")

    except Exception as _e:
        print(f"  py-feat error: {_e}")
        f_vec = None
else:
    print("No video/image provided — zero vector.")

if f_vec is None:
    print("  Video branch -> zero vector.")
    f_vec = np.zeros(20, dtype="float32")

np.save("/tmp/f_vec.npy", f_vec)
print(f"\nVideo vector -> /tmp/f_vec.npy  {f_vec.shape}  OK")
print("Run STEP E to get the prediction.")


/tmp/ipykernel_3239/429234216.py:24: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, _attr):
/tmp/ipykernel_3239/429234216.py:24: FutureWarning: In the future `np.str` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, _attr):


Installing av ...
Installing py-feat==0.5.1 (~30s) ...
  done ✅
  py-feat error: Detector unavailable: No module named 'lib2to3'
  Video branch -> zero vector.

Video vector -> /tmp/f_vec.npy  (20,)  OK
Run STEP E to get the prediction.


In [ ]:
# ── STEP E: Predict ─────────────────────────────────────────────
import numpy as np, tensorflow as tf

MODEL_PATH = str(FYP_MODEL_PATH if "FYP_MODEL_PATH" in globals() else "/content/demo/fusion_model.keras")

def focal_loss(gamma=2.0, alpha=0.25):
    def loss_fn(y_true, y_pred):
        y_true  = tf.cast(y_true, tf.float32)
        y_pred  = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        ce      = -(y_true*tf.math.log(y_pred)+(1-y_true)*tf.math.log(1-y_pred))
        p_t     = y_true*y_pred+(1-y_true)*(1-y_pred)
        alpha_t = y_true*alpha+(1-y_true)*(1-alpha)
        return tf.reduce_mean(alpha_t*tf.pow(1-p_t,gamma)*ce)
    return loss_fn

t_vec = np.load("/tmp/t_vec.npy")
a_vec = np.load("/tmp/a_vec.npy")
f_vec = np.load("/tmp/f_vec.npy")
print(f"Vectors: text {t_vec.shape}, audio {a_vec.shape}, video {f_vec.shape}")

model = tf.keras.models.load_model(
    MODEL_PATH, custom_objects={"loss_fn": focal_loss()}, compile=False)

prob = float(model.predict(
    [a_vec.reshape(1,-1), f_vec.reshape(1,-1), t_vec.reshape(1,-1)], verbose=0
).ravel()[0])

predicted = "DEPRESSED"  if prob >= 0.5 else "NOT DEPRESSED"
conf      = prob*100     if prob >= 0.5 else (1-prob)*100
icon      = "RED" if prob >= 0.5 else "GREEN"

print()
print("=" * 54)
print("     TRI-DEP  -  PREDICTION  RESULT")
print("=" * 54)
print(f"  Depression probability :  {prob*100:5.1f}%")
print(f"  Prediction             :  [{icon}]  {predicted}")
print(f"  Confidence             :  {conf:5.1f}%")
print("=" * 54)

_BAR  = 44
_fill = int(round(prob * _BAR))
_bar  = "#" * _fill + "-" * (_BAR - _fill)
print(f"\n  0%  [{_bar}]  100%")
print(f"       {'':>{_fill}}^")
print(f"  (threshold 50% -- above = Depressed, below = Not Depressed)")


Vectors: text (768,), audio (1536,), video (20,)

     TRI-DEP  -  PREDICTION  RESULT
  Depression probability :    4.6%
  Prediction             :  [GREEN]  NOT DEPRESSED
  Confidence             :   95.4%

  0%  [##------------------------------------------]  100%
         ^
  (threshold 50% -- above = Depressed, below = Not Depressed)


---
## Optional — Streamlit web app
Run the two cells below to launch the interactive web interface.

In [ ]:
# ── STEP 1: install packages ──────────────────────────────────────────────────
import subprocess, sys

# Remove conflicting feat from environment BEFORE any imports
subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'feat'],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'py-feat', 'gradio',
                       'sentence-transformers'],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print('Packages installed ✅')

# ── STEP 2: copy main.py from Drive ───────────────────────────────────────────
import shutil, os
DRIVE = '/content/drive/MyDrive/DAIC'
src   = f'{DRIVE}/main.py'
if os.path.exists(src):
    shutil.copy(src, '/content/main.py')
    print(f'main.py copied from Drive ✅')
else:
    print(f'WARNING: {src} not found — InducT-GCN will be unavailable')
    print('Upload main.py to MyDrive/DAIC/ and re-run this cell')

# ── STEP 3: verify py-feat by clearing module cache ───────────────────────────
import importlib

# Force-remove any cached feat modules from previous imports
for mod in list(sys.modules.keys()):
    if mod == 'feat' or mod.startswith('feat.'):
        del sys.modules[mod]
importlib.invalidate_caches()

try:
    from feat import Detector
    print('py-feat import verified ✅')
except Exception as e:
    print(f'py-feat import test failed: {e}')
    print('Video features will use zero vector')

Packages installed ✅
main.py copied from Drive ✅
py-feat import test failed: No module named 'lib2to3'
Video features will use zero vector


In [ ]:
import sys, os, gc, importlib, traceback, subprocess
sys.path.insert(0, '/content')

import gradio as gr
import numpy as np
import pandas as pd
import tensorflow as tf
import torch

from pathlib import Path
base_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

if Path('/content/drive/MyDrive/DAIC').exists():
    DRIVE = '/content/drive/MyDrive/DAIC'
    FYP_MODEL_PATH = '/content/demo/fusion_model.keras'
    DATA_NPZ_PATH = '/content/demo/demo_data.npz'
    MERGED_CSV_PATH = f'{DRIVE}/fusion_results/merged_predictions.csv'
else:
    DRIVE = str(base_dir / 'saved_models')
    FYP_MODEL_PATH = str(base_dir / 'saved_models' / 'demo_bundle' / 'fusion_model.keras')
    DATA_NPZ_PATH = str(base_dir / 'saved_models' / 'demo_bundle' / 'demo_data.npz')
    MERGED_CSV_PATH = str(base_dir / 'saved_models' / 'merged_predictions.csv')

IGCN_MODEL = f'{DRIVE}/model/Participant/9_induct-gcn[original-features250]/model_inductgcn[250].pkl'
IGCN_VTZER = f'{DRIVE}/model/Participant/9_induct-gcn[original-features250]/vtzer_inductgcn[250].pkl'

def focal_loss(gamma=2.0, alpha=0.25):
    def loss_fn(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1-1e-7)
        ce  = -(y_true*tf.math.log(y_pred)+(1-y_true)*tf.math.log(1-y_pred))
        p_t = y_true*y_pred+(1-y_true)*(1-y_pred)
        a_t = y_true*alpha+(1-y_true)*(1-alpha)
        return tf.reduce_mean(a_t*tf.pow(1-p_t,gamma)*ce)
    return loss_fn

print('Loading FYP model...')
fyp_model = tf.keras.models.load_model(FYP_MODEL_PATH,
    custom_objects={'loss_fn': focal_loss()}, compile=False)
data  = np.load(DATA_NPZ_PATH, allow_pickle=True)
A, F, T, Y, PIDS = data['A'], data['F'], data['T'], data['Y'], data['PIDS']
pid_list = [int(p) for p in PIDS]
try:
    merged   = pd.read_csv(MERGED_CSV_PATH)
    dev_pids = list(merged['participant_id'].astype(int))
except:
    merged = None; dev_pids = []

# ── InducT-GCN ────────────────────────────────────────────────────────────────
print('Installing optuna (required by main.py)...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'optuna', '-q'],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print('Loading InducT-GCN...')
igcn = None
if not os.path.exists('/content/main.py'):
    print('  ⚠️  /content/main.py not found — run STEP 1 cell first')
elif not os.path.exists(IGCN_MODEL):
    print(f'  ⚠️  Model file not found:\n      {IGCN_MODEL}')
elif not os.path.exists(IGCN_VTZER):
    print(f'  ⚠️  Vectorizer not found:\n      {IGCN_VTZER}')
else:
    try:
        from main import load_induct_gcn_model
        import main as _main_mod
        # DEVICE is only set when main.py runs as __main__ — set it manually
        _main_mod.DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        igcn = load_induct_gcn_model(IGCN_MODEL, IGCN_VTZER, device='cpu')
        igcn.eval()
        print(f'  InducT-GCN ready ✅  classes={list(igcn.classes_)}')
    except Exception as e:
        traceback.print_exc()
        print(f'  InducT-GCN failed: {e}')

print('Loading SBERT...')
from sentence_transformers import SentenceTransformer
sbert = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

print('Loading wav2vec2...')
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor
_w2v_fe  = Wav2Vec2FeatureExtractor.from_pretrained('facebook/wav2vec2-base-960h')
_w2v_mdl = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base-960h').eval()

print('Loading py-feat...')
_PYFEAT_DIR = '/tmp/pyfeat_install'
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', 'py-feat==0.5.1',
     f'--target={_PYFEAT_DIR}', '-q'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
if _PYFEAT_DIR not in sys.path:
    sys.path.insert(0, _PYFEAT_DIR)
for _m in list(sys.modules.keys()):
    if _m == 'feat' or _m.startswith('feat.'):
        del sys.modules[_m]
importlib.invalidate_caches()

def _get_detector():
    try:
        from feat import Detector; return Detector
    except Exception: pass
    try:
        return importlib.import_module('feat.detector').Detector
    except Exception as _e:
        raise ImportError(f"Detector unavailable: {_e}")

PYFEAT_OK = False
try:
    _D = _get_detector(); PYFEAT_OK = True; print('  py-feat ready ✅')
except Exception as e:
    print(f'  py-feat unavailable: {e}')

print('\nAll models loaded ✅')

# ── Gradio 3.x / 4.x file compat ─────────────────────────────────────────────
def _fpath(f):
    if f is None: return None
    if isinstance(f, str): return f
    if hasattr(f, 'path'): return f.path
    if hasattr(f, 'name'): return f.name
    if isinstance(f, dict): return f.get('path') or f.get('name') or f.get('tmp_path')
    return str(f)

def _fname(f):
    if f is None: return ''
    if isinstance(f, str): return f
    if hasattr(f, 'orig_name'): return f.orig_name or _fpath(f)
    if isinstance(f, dict): return f.get('orig_name') or f.get('name') or ''
    return _fpath(f) or ''

# ── HTML helpers ──────────────────────────────────────────────────────────────
def make_card(label, prob, true_label=None):
    pred = prob >= 0.5

    color  = '#ff1744' if pred else '#00e676'
    bg     = '#fff1f2' if pred else '#ecfff4'
    border = '#ff4569' if pred else '#00c853'

    emoji = '🔴' if pred else '🟢'
    txt   = 'Depressed' if pred else 'Not Depressed'
    conf  = prob*100 if pred else (1-prob)*100

    h = f"""
    <div style="
        background:{bg};
        border:3px solid {border};
        border-radius:18px;
        padding:1rem;
        box-shadow:0 10px 25px rgba(0,0,0,.15);
        text-align:center;">

        <div style="
            font-size:.8rem;
            color:#2563eb;
            font-weight:700;
            text-transform:uppercase;">
            {label}
        </div>

        <div style="font-size:3rem;">{emoji}</div>

        <div style="
            font-weight:800;
            color:{color};
            font-size:1.4rem;">
            {txt}
        </div>

        <div style="
            color:#1e40af;
            font-size:.9rem;">
            Probability: <b>{prob*100:.1f}%</b>
            &nbsp;|&nbsp;
            Confidence: <b>{conf:.1f}%</b>
        </div>
    </div>
    """

    if true_label is not None:
        match = "✅" if pred==(true_label==1) else "❌"
        h += f"""
        <p style="
            text-align:center;
            color:#2563eb;
            font-weight:600;
            margin-top:.5rem;">
            {match} Ground Truth:
            {"Depressed" if true_label==1 else "Not Depressed"}
        </p>
        """

    return h

def make_final(or_pred, true_label=None):

    color  = '#ff1744' if or_pred else '#00e676'
    border = color

    bg = (
        "linear-gradient(135deg,#fff5f5,#ffd6d6,#ffeaea)"
        if or_pred else
        "linear-gradient(135deg,#f1fff6,#d9ffe8,#ffffff)"
    )

    emoji = "🔴" if or_pred else "🟢"
    txt   = "DEPRESSED" if or_pred else "NOT DEPRESSED"

    h = f"""
    <div style="
        background:{bg};
        border:4px solid {border};
        border-radius:22px;
        padding:2rem;
        text-align:center;
        box-shadow:0 15px 35px rgba(0,0,0,.18);
        margin-top:1rem;">

        <div style="font-size:4rem;">{emoji}</div>

        <div style="
            font-size:2.3rem;
            font-weight:900;
            color:{color};">
            {txt}
        </div>

        <div style="
            color:#1e40af;
            font-weight:700;
            margin-top:.5rem;">
            Combined Ensemble — OR Vote
        </div>

    </div>
    """

    if true_label is not None:
        match = "✅ Correct" if or_pred==(true_label==1) else "❌ Mismatch"
        h += f"""
        <p style="
            text-align:center;
            color:#2563eb;
            font-weight:700;">
            <b>Ground Truth:</b>
            {"Depressed" if true_label==1 else "Not Depressed"}
            {match}
        </p>
        """

    return h

# ── parse DAIC-WOZ transcript ─────────────────────────────────────────────────
def parse_transcript(filepath, orig_name):
    if str(orig_name).lower().endswith('.csv'):
        try:
            for use_header in ('infer', None):
                df = pd.read_csv(filepath, header=use_header, sep=None, engine='python')
                df.columns = [str(c).strip().lower() for c in df.columns]
                cols = list(df.columns)

                sc = next((c for c in cols if 'speaker' in c), None)
                vc = next((c for c in cols if c in ('value','text','utterance','transcript')), None)

                if sc is None:
                    for col in cols:
                        vals = df[col].astype(str).str.strip().str.lower().unique()
                        if any(v in ('ellie', 'participant') for v in vals):
                            sc  = col
                            idx = cols.index(col)
                            vc  = cols[idx+1] if idx+1 < len(cols) else None
                            break

                if sc and vc:
                    mask = df[sc].astype(str).str.strip().str.lower() == 'participant'
                    text = ' '.join(df.loc[mask, vc].dropna().astype(str).tolist())
                    if text.strip():
                        return text, f'✅ {mask.sum()} participant turns extracted — Ellie filtered out'

            text = ' '.join(df.iloc[:, -1].dropna().astype(str).tolist())
            return text, '⚠️ Speaker column not found — all rows joined'
        except Exception as e:
            with open(filepath, 'r', errors='ignore') as f: text = f.read()
            return text, f'⚠️ CSV parse failed ({e})'
    with open(filepath, 'r', errors='ignore') as f: text = f.read()
    return text, '✅ Plain text transcript loaded'

# ── feature extractors ────────────────────────────────────────────────────────
def text_to_vec(text):
    import re
    text  = re.sub(r'\s+', ' ', text).strip()
    sents = [s.strip() for s in text.replace('?','.').split('.') if s.strip()]
    if not sents: sents = [text or 'no speech']
    return sbert.encode(sents, show_progress_bar=False, convert_to_numpy=True).mean(0).astype('float32')

def audio_to_vec(audio_path):
    import librosa
    SR, WIN, MAX_W = 16000, 16000*8, 20
    sp, sr = librosa.load(audio_path, sr=None, mono=True, duration=MAX_W*8+30)
    if sr != SR: sp = librosa.resample(sp.astype('float32'), orig_sr=sr, target_sr=SR)
    wins = [sp[i:i+WIN] for i in range(0, len(sp)-WIN+1, WIN)][:MAX_W]
    if not wins: return np.zeros(1536, dtype='float32')
    with torch.no_grad():
        inp = _w2v_fe(wins, sampling_rate=SR, return_tensors='pt', padding=True)
        hid = _w2v_mdl(inp.input_values).last_hidden_state
        emb = hid.mean(dim=1).cpu().numpy()
    return np.concatenate([emb.mean(0), emb.std(0)]).astype('float32')

def video_to_vec(video_path):
    if not PYFEAT_OK:
        return np.zeros(20,'float32'), '⚠️ py-feat unavailable — zero vector used'
    try:
        Detector = _get_detector()
        COLS = ['AU01_r','AU02_r','AU04_r','AU05_r','AU06_r','AU09_r','AU10_r',
                'AU12_r','AU14_r','AU15_r','AU17_r','AU20_r','AU25_r','AU26_r',
                'AU04_c','AU12_c','AU15_c','AU23_c','AU28_c','AU45_c']
        MAP  = {'AU01_r':'AU01','AU02_r':'AU02','AU04_r':'AU04','AU05_r':'AU05',
                'AU06_r':'AU06','AU09_r':'AU09','AU10_r':'AU10','AU12_r':'AU12',
                'AU14_r':'AU14','AU15_r':'AU15','AU17_r':'AU17','AU20_r':'AU20',
                'AU25_r':'AU25','AU26_r':'AU26','AU04_c':'AU04','AU12_c':'AU12',
                'AU15_c':'AU15','AU23_c':'AU23','AU28_c':'AU28','AU45_c':'AU43'}
        det = Detector(device='cuda' if torch.cuda.is_available() else 'cpu')
        fea = det.detect_video(video_path)
        au  = fea.aus.dropna(how='all').reset_index(drop=True)
        if len(au)==0: raise ValueError('No face detected')
        arr = np.zeros((len(au),20),'float32')
        for i,tc in enumerate(COLS):
            pc = MAP.get(tc)
            if pc and pc in au.columns: arr[:,i] = au[pc].fillna(0).values.astype('float32')
        if arr.shape[0]>1: arr=(arr-arr.mean(0,keepdims=True))/(arr.std(0,keepdims=True)+1e-8)
        return arr.mean(0).astype('float32'), f'✅ Video: {len(au)} frames with face detected'
    except Exception as e:
        return np.zeros(20,'float32'), f'⚠️ Video failed: {e}'

def igcn_prob(text):
    if igcn is None: return None
    try:
        probs   = igcn.predict_proba(text)   # 1-D numpy array (n_classes,)
        classes = list(igcn.classes_)
        dep_idx = next((i for i, c in enumerate(classes)
                        if str(c) in ('1', 'positive', 'depressed')), len(classes)-1)
        return float(probs[dep_idx])
    except Exception as e:
        print(f'  igcn_prob error: {e}')
        traceback.print_exc()
        return None

# ── callbacks ─────────────────────────────────────────────────────────────────
def cb_known(chosen_id):
    chosen_id = int(chosen_id)
    idx = pid_list.index(chosen_id)
    fp  = float(fyp_model.predict([A[idx:idx+1],F[idx:idx+1],T[idx:idx+1]],verbose=0).ravel()[0])
    tl  = int(Y[idx])
    row = merged[merged['participant_id']==chosen_id] if merged is not None and chosen_id in dev_pids else None
    c1  = make_card('🎵 FYP Model (A+V+T)', fp, tl)
    if row is not None and not row.empty:
        tp   = float(row['text_prob'].values[0])
        c2   = make_card('📝 InducT-GCN (Text)', tp, tl)
        or_r = (fp>=0.5) or (tp>=0.5)
        c3   = make_card('🔗 Combined OR vote', 1.0 if or_r else 0.0, tl)
        fin  = make_final(or_r, tl)
    else:
        c2  = '<p style="color:#94a3b8;padding:1rem;">InducT-GCN predictions not in merged_predictions.csv</p>'
        c3  = '<p style="color:#94a3b8;padding:1rem;">—</p>'
        fin = make_final(fp>=0.5, tl)
    return c1, c2, c3, fin

def cb_live(audio_file, video_file, text_file):
    try:
        audio_path = _fpath(audio_file)
        video_path = _fpath(video_file)
        text_path  = _fpath(text_file)
        orig_name  = _fname(text_file)

        if audio_path is None: return 'ERROR: Audio required','','','',''
        if text_path  is None: return 'ERROR: Transcript required','','','',''

        log = ''
        raw_text, info = parse_transcript(text_path, orig_name)
        log += info + '\n'
        log += f'Preview: {raw_text[:200]}...\n\n'
        if not raw_text.strip(): return 'ERROR: Transcript empty','','','',''

        t_vec = text_to_vec(raw_text);    log += f'✅ Text (SBERT): {t_vec.shape}\n'
        a_vec = audio_to_vec(audio_path); log += f'✅ Audio (wav2vec2): {a_vec.shape}\n'

        if video_path:
            f_vec, vmsg = video_to_vec(video_path); log += vmsg+'\n'
        else:
            f_vec = np.zeros(20,'float32'); log += '⚠️ No video — zero vector\n'

        fp = float(fyp_model.predict(
            [a_vec.reshape(1,-1), f_vec.reshape(1,-1), t_vec.reshape(1,-1)],
            verbose=0).ravel()[0])
        log += f'✅ FYP probability: {fp*100:.1f}%\n'

        ip = igcn_prob(raw_text)
        log += (f'✅ InducT-GCN: {ip*100:.1f}%\n' if ip is not None
                else '⚠️ InducT-GCN not available\n')

        or_r = (fp>=0.5) or (ip>=0.5 if ip is not None else False)
        c1 = make_card('🎵 FYP Model (A+V+T)', fp)
        c2 = (make_card('📝 InducT-GCN (Text)', ip) if ip is not None
              else '<p style="color:#94a3b8;padding:1rem;">InducT-GCN not available</p>')
        c3 = make_card('🔗 Combined OR vote', 1.0 if or_r else 0.0)
        return log, c1, c2, c3, make_final(or_r)

    except Exception as e:
        tb = traceback.format_exc()
        print(tb)
        return f'ERROR: {e}\n\n{tb}', '', '', '', ''

# ── UI ────────────────────────────────────────────────────────────────────────
perf_df = pd.DataFrame({
    'Model'          :['FYP Audio only','FYP Video only','FYP Text (SBERT)',
                       'FYP Fused (A+V+T)','InducT-GCN Text','Combined OR vote'],
    'Accuracy'       :[0.513,0.481,0.693,0.714,0.800,0.800],
    'Precision (dep)':[0.300,0.281,0.482,0.600,0.667,0.647],
    'Recall (dep)'   :[0.482,0.482,0.482,0.500,0.833,0.917],
    'F1 (dep)'       :[0.370,0.355,0.482,0.545,0.741,0.759],
    'F1 (weighted)'  :[0.534,0.504,0.693,0.707,0.804,0.805],
})

with gr.Blocks(title='TriDep — Depression Screening') as demo:
    gr.HTML("""
<div style="
background:linear-gradient(135deg,#2563eb,#7c3aed,#ec4899);
padding:20px;
border-radius:18px;
text-align:center;
box-shadow:0 10px 25px rgba(0,0,0,.2);
margin-bottom:12px;">

<div style="
font-size:2rem;
font-weight:800;
color:white;">
🧠 TriDep — Multi-Modal Depression Screening
</div>

<div style="
color:#f8fafc;
font-size:.9rem;
font-weight:600;
letter-spacing:.08em;
text-transform:uppercase;">
Audio · Video · Text (InducT-GCN) · Combined Ensemble
</div>

</div>
""")

    with gr.Tab("🔬 Known Subject"):
        gr.Markdown("Select a DAIC-WOZ dev set participant to see all three model predictions.")
        sub_dd = gr.Dropdown(choices=[str(p) for p in (dev_pids if dev_pids else pid_list)],
                             value=str(dev_pids[0]) if dev_pids else str(pid_list[0]), label="Subject ID")
        btn1 = gr.Button("🔬 Run Screening", variant="primary")
        with gr.Row(): o1a=gr.HTML(); o1b=gr.HTML(); o1c=gr.HTML()
        o1f = gr.HTML()
        btn1.click(cb_known, inputs=[sub_dd], outputs=[o1a,o1b,o1c,o1f])

    with gr.Tab("📤 Live Upload"):
        gr.Markdown("Upload audio + DAIC-WOZ transcript (`.csv` or `.txt`). Video optional.")
        with gr.Row():
            in_au = gr.File(label="🎵 Audio (.wav/.mp3)", file_types=['.wav','.mp3','.ogg','.m4a'])
            in_vi = gr.File(label="🎥 Video — optional (.mp4)", file_types=['.mp4','.avi','.mov'])
            in_tx = gr.File(label="📝 Transcript (.csv/.txt)", file_types=['.csv','.txt'])
        btn2  = gr.Button("🔬 Run Live Screening", variant="primary")
        o_log = gr.Textbox(label="Processing log", lines=10, interactive=False)
        with gr.Row(): o2a=gr.HTML(); o2b=gr.HTML(); o2c=gr.HTML()
        o2f = gr.HTML()
        btn2.click(cb_live, inputs=[in_au,in_vi,in_tx], outputs=[o_log,o2a,o2b,o2c,o2f])

    with gr.Tab("📊 Model Performance"):
        gr.Markdown("### System Performance — Dev Set (35 subjects)")
        gr.Dataframe(value=perf_df, interactive=False)
        gr.Markdown("""
**✅ OR vote catches 91.7% of depressed patients — highest recall.**

**Methodology:**
- **FYP model**: audio prosody (wav2vec2) + facial AUs (CLNF) + SBERT text
- **InducT-GCN**: depression markers via word co-occurrence graph
- **OR vote**: flag if *either* model detects depression — maximises recall
        """)
    gr.HTML("""
<hr style="border-color:#60a5fa;">

<p style="
text-align:center;
background:#eff6ff;
padding:12px;
border-radius:10px;
font-size:.9rem;
font-weight:600;
color:#1d4ed8;">

⚠️ Research Prototype — Not Intended for Clinical Diagnosis

</p>
""")

demo.launch(share=True)


Loading FYP model...
Installing optuna (required by main.py)...
Loading InducT-GCN...
  InducT-GCN ready ✅  classes=['negative', 'positive']
Loading SBERT...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading wav2vec2...


Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading py-feat...
  py-feat unavailable: Detector unavailable: No module named 'lib2to3'

All models loaded ✅
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cba6a740f77a08b647.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
